# Siding Repair: Subtopic Updater

For a given cluster stub, generates content for every `*Content to be generated.*` subtopic section
and writes the results back to the file — including a consolidated `## References` table.

Each generation call receives:
- The **full document** as context (so the writer sees the page's scope and existing sections)
- **Top-K RAG passages** from the construction books LanceDB

Superscript references are renumbered globally across all subtopics before writing.

In [ ]:
from pathlib import Path

LANCEDB_PATH = r"C:\Users\tfalcon\microsites\tools\programatic writing stuffs\DBA writer\lancedb_construction_books"
LANCEDB_TABLE = "construction_books"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
MODEL = "gpt-4o"
TEMPERATURE = 0.7
TOP_K = 5
WORD_TARGET = 200

# Target file — change these to process a different cluster
TARGET_SLUG = "hardie-fiber-cement-siding-repair"
TARGET_LOCATION = "portland"

SIDING_REPAIR_DIR = Path("../../../apps/siding-repair/src/data/generated_content")
AGENTS_DIR = Path("../agents")
CONTENT_REVIEW_DIR = Path("../../content-review")

In [ ]:
import sys, os, re
from dotenv import load_dotenv

load_dotenv("../.env")
sys.path.insert(0, str(CONTENT_REVIEW_DIR.resolve()))

import lancedb
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from agent_loader import load_agent

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
embedder = SentenceTransformer(EMBEDDING_MODEL)
db = lancedb.connect(LANCEDB_PATH)
table = db.open_table(LANCEDB_TABLE)
agent = load_agent(AGENTS_DIR / "01-subtopic-writer.md")
print(f"LanceDB: {table.count_rows()} docs | Agent: {agent.name}")

In [ ]:
# Find and load the target stub file
filename = f"service_page_cluster_{TARGET_SLUG}_{TARGET_LOCATION}.md"
stub_path = SIDING_REPAIR_DIR / filename

if not stub_path.exists():
    raise FileNotFoundError(f"Stub not found: {stub_path}")

doc_text = stub_path.read_text(encoding="utf-8")

# Parse subtopics from CLUSTER_META
meta_match = re.search(r'<!--\s*CLUSTER_META([\s\S]*?)-->', doc_text)
if not meta_match:
    raise ValueError("No CLUSTER_META block found")

subtopics = []
in_subtopics = False
for line in meta_match.group(1).split('\n'):
    s = line.strip()
    if s.startswith('subtopics:'):
        in_subtopics = True; continue
    if in_subtopics:
        if s.startswith('- '): subtopics.append(s[2:].strip())
        elif s and not s.startswith('-'): in_subtopics = False

# Identify which subtopics still need content
pending = [t for t in subtopics if f"## {t}\n*Content to be generated.*" in doc_text]

print(f"File: {filename}")
print(f"Subtopics total: {len(subtopics)} | Pending: {len(pending)}")
for t in subtopics:
    status = "PENDING" if t in pending else "done"
    print(f"  [{status}] {t}")

In [ ]:
def generate_subtopic(subtopic: str, doc_text: str, service: str, location_full: str) -> tuple[str, list[str]]:
    """
    Returns (content_body, ref_rows) where:
      content_body  — markdown section body with <sup>1</sup> / <sup>2</sup>
      ref_rows      — list of raw markdown table rows: ["| 1 | ... |", "| 2 | ... |"]
    """
    # RAG retrieval
    query = f"{subtopic} {service} {location_full}"
    results = table.search(embedder.encode([query])[0]).limit(TOP_K).to_pandas()
    rag_context = "\n\n---\n\n".join(
        f"Source: {row.get('source','?')} (Page {row.get('page','N/A')})\n{row['text']}"
        for _, row in results.iterrows()
    )

    user_prompt = f"""## Full document context (read before writing — understand the page scope and other sections):

{doc_text}

---

## Construction reference material (RAG):

{rag_context}

---

Write a ~{WORD_TARGET}-word section body for:

Subtopic: {subtopic}
Parent page: {service} — {location_full}

Rules:
- Do NOT include the section heading
- Use <sup>1</sup> and <sup>2</sup> for inline citations
- After the section body, output exactly one line: <!-- REFS -->
- Then output exactly 2 pipe-delimited reference rows (no header row), format:
  | N | Source Title | p. X | Brief note |
- Return nothing else"""

    response = client.chat.completions.create(
        model=MODEL,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": agent.system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )
    raw = response.choices[0].message.content.strip()

    # Split on <!-- REFS -->
    parts = raw.split("<!-- REFS -->", 1)
    content_body = parts[0].strip()
    ref_rows = []
    if len(parts) == 2:
        ref_rows = [l.strip() for l in parts[1].strip().splitlines() if l.strip().startswith("|")]

    return content_body, ref_rows

In [ ]:
location_full = "Portland, Oregon" if TARGET_LOCATION == "portland" else "Seattle, Washington"

# Re-read doc fresh so we always pass the current file state
doc_text = stub_path.read_text(encoding="utf-8")

# service label from meta block
service_match = re.search(r'service:\s*(.+)', doc_text)
service_label = service_match.group(1).strip() if service_match else "siding repair"

results_store = {}  # subtopic -> (content_body, ref_rows)

for i, subtopic in enumerate(pending):
    print(f"[{i+1}/{len(pending)}] Generating: {subtopic} ...", end=" ", flush=True)
    content, refs = generate_subtopic(subtopic, doc_text, service_label, location_full)
    results_store[subtopic] = (content, refs)
    words = len(content.split())
    print(f"{words} words, {len(refs)} refs")

print("\nDone.")

In [ ]:
# Preview all generated content before writing
for subtopic, (content, refs) in results_store.items():
    print(f"\n{'='*60}")
    print(f"## {subtopic}")
    print(f"{'='*60}")
    print(content)
    if refs:
        print("\n--- refs ---")
        for r in refs:
            print(r)

In [ ]:
def renumber_refs(text: str, offset: int) -> str:
    """Shift <sup>1</sup> and <sup>2</sup> by offset."""
    text = re.sub(r'<sup>1</sup>', f'<sup>{offset + 1}</sup>', text)
    text = re.sub(r'<sup>2</sup>', f'<sup>{offset + 2}</sup>', text)
    return text

def renumber_ref_rows(rows: list[str], offset: int) -> list[str]:
    """Renumber the leading | N | in each ref row."""
    out = []
    for row in rows:
        # Replace the first cell number: | 1 | ... or | 2 | ...
        row = re.sub(r'^\|\s*\d+\s*\|', f'| {offset + len(out) + 1} |', row)
        out.append(row)
    return out

updated = stub_path.read_text(encoding="utf-8")
all_ref_rows = []
global_offset = 0

for subtopic, (content, refs) in results_store.items():
    # Renumber inline superscripts
    numbered_content = renumber_refs(content, global_offset)
    # Renumber reference rows
    numbered_refs = renumber_ref_rows(refs, global_offset)
    all_ref_rows.extend(numbered_refs)
    global_offset += len(refs)

    # Replace the stub placeholder
    placeholder = f"## {subtopic}\n*Content to be generated.*"
    replacement = f"## {subtopic}\n{numbered_content}"
    updated = updated.replace(placeholder, replacement)

# Build references table
if all_ref_rows:
    ref_table = (
        "\n## References\n\n"
        "| # | Source | Page | Note |\n"
        "|---|--------|------|------|\n"
        + "\n".join(all_ref_rows)
        + "\n"
    )
    # Insert before ## Page Metadata, or append at end
    if "## Page Metadata" in updated:
        updated = updated.replace("## Page Metadata", ref_table + "\n## Page Metadata")
    else:
        updated += ref_table

# Mark status as draft
updated = re.sub(r'(status:\s*)stub', r'\1draft', updated)

stub_path.write_text(updated, encoding="utf-8")
print(f"Written: {stub_path.name}")
print(f"Sections filled: {len(results_store)} | References added: {len(all_ref_rows)}")

In [ ]:
# Verify — print the updated file
print(stub_path.read_text(encoding="utf-8"))